In [1]:
import pandas as pd
import requests

/Users/celine/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:

def get_shopping_malls():
    overpass_url = "https://overpass-api.de/api/interpreter"

    # Singapore admin boundary + malls
    query = """
    [out:json][timeout:120];
    area["ISO3166-1"="SG"][admin_level=2]->.sg;
    (
      node["shop"="mall"](area.sg);
      way["shop"="mall"](area.sg);
      relation["shop"="mall"](area.sg);
      node["building"="mall"](area.sg);
      way["building"="mall"](area.sg);
      relation["building"="mall"](area.sg);
    );
    out center tags;
    """

    r = requests.post(overpass_url, data={"data": query}, timeout=180)
    r.raise_for_status()
    data = r.json()

    rows = []
    for e in data.get("elements", []):
        tags = e.get("tags", {})
        name = tags.get("name")
        if not name:
            continue

        # node has lat/lon; way/relation has center
        lat = e.get("lat", e.get("center", {}).get("lat"))
        lon = e.get("lon", e.get("center", {}).get("lon"))

        if lat is not None and lon is not None:
            rows.append({"mall_name": name, "lat": lat, "lon": lon})

    mall_df = pd.DataFrame(rows).drop_duplicates(subset=["mall_name"]).reset_index(drop=True)
    return mall_df



In [3]:
mall_df = get_shopping_malls()
mall_df.head()

,mall_name,lat,lon
0,The Star Vista,1.308002,103.788382
1,Bencoolen Underground Mall,1.298181,103.849647
2,Katong V,1.303133,103.903231
3,The Poiz Centre,1.331436,103.868571
4,Excelsior Shopping Centre,1.292099,103.849689


In [8]:
mall_df.to_csv("../../data/cleaned/mall_cleaned.csv")